# 03 — 描述统计、可视化与分析

本 Notebook 完成以下任务：
1. 计算 13 个财务指标的年度描述统计（均值、中位数、标准差、极值）
2. 绘制 Lev 均值/中位数时序图和 ROA/Cash 均值时序图
3. 分析 7 大行业的算术平均和加权平均负债率
4. 比较两种加权方式的差异
5. 绘制 Top1 箱线图并分析股权结构变化
6. 回答股权分置改革、股权分散化等相关问题

使用数据：`data/clean/firm_year_clean.csv`（71,787 观测，5,820 家公司）

---
## 4. 描述统计与可视化


## 4.1 年度描述统计

对以下 13 个变量按年度计算均值、中位数、标准差、最小值、最大值和样本量：
`Lev, SL, LL, SDR, Cash, ROA, ROE, SLoan, LLoan, Top1, HHI5, Size, Age`
其中 Lev–HHI5 使用缩尾后版本，Top1/Size/Age 使用原始值。
结果保存为 `yearly_summary.csv` 和 `yearly_summary.xlsx`。


In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm.fontManager.addfont(r'C:\Windows\Fonts\simhei.ttf')
plt.rcParams.update({
    'figure.dpi': 120, 'figure.figsize': (10, 4.5),
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
    'font.family': 'sans-serif',
    'font.sans-serif': ['SimHei'],
    'axes.unicode_minus': False,
    'savefig.dpi': 200, 'savefig.bbox': 'tight',
})

BASE = Path.cwd()
CLEAN = BASE / 'data' / 'clean'
OUT_TABLES = BASE / 'output' / 'tables'
OUT_FIGS = BASE / 'output' / 'figures'
for d in [OUT_TABLES, OUT_FIGS]:
    d.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CLEAN / 'firm_year_clean.csv', dtype={'code': str})
df['year'] = pd.to_numeric(df['year'], errors='coerce')
print(f'数据加载完成：{len(df)} 行，{df["code"].nunique()} 家公司')
print(f'年份范围：{int(df["year"].min())} – {int(df["year"].max())}')


### 变量中文名映射


In [ ]:
VAR_CN = {
    'Lev': '总负债率', 'SL': '流动负债率', 'LL': '长期负债率',
    'SDR': '短债比率', 'Cash': '现金比率',
    'ROA': '总资产收益率', 'ROE': '净资产收益率',
    'SLoan': '短期借款率', 'LLoan': '长期借款率',
    'Top1': '第一大股东持股比', 'HHI5': '股权集中度',
    'Size': '公司规模', 'Age': '上市年限',
}


In [ ]:
var_cols = [
    ('Lev', 'Lev'), ('SL', 'SL'), ('LL', 'LL'), ('SDR', 'SDR'),
    ('Cash', 'Cash'), ('ROA', 'ROA'), ('ROE', 'ROE'),
    ('SLoan', 'SLoan'), ('LLoan', 'LLoan'),
    ('Top1', 'Top1_raw'), ('HHI5', 'HHI5'),
    ('Size', 'Size_raw'), ('Age', 'Age_raw'),
]

summary_rows = []
for vname, vcol in var_cols:
    g = df.groupby('year')[vcol]
    s = g.agg(['mean', 'median', 'std', 'min', 'max', 'count']).reset_index()
    s.insert(0, 'variable', vname)
    summary_rows.append(s)

yearly = pd.concat(summary_rows, ignore_index=True)
yearly.columns = ['variable', 'year', 'mean', 'median', 'std', 'min', 'max', 'n']
yearly.to_csv(OUT_TABLES / 'yearly_summary.csv', index=False, encoding='utf-8-sig')
yearly.to_excel(OUT_TABLES / 'yearly_summary.xlsx', index=False)
print(f'年度描述统计已保存：{len(yearly)} 行')
print(yearly.head(13).to_string(index=False))


### 预计算关键数字供图 1、图 2 分析引用


In [ ]:
yr_range = sorted(df['year'].dropna().unique())
first_yr, last_yr = int(yr_range[0]), int(yr_range[-1])
n_years = len(yr_range)

n_per_yr = df.groupby('year').size()
min_n_year = int(n_per_yr.idxmin())
max_n_year = int(n_per_yr.idxmax())
min_n = int(n_per_yr.min())
max_n = int(n_per_yr.max())

lev_mean = df.groupby('year')['Lev'].mean()
lev_first = lev_mean.iloc[0]
lev_last = lev_mean.iloc[-1]
lev_min_yr = int(lev_mean.idxmin())
lev_max_yr = int(lev_mean.idxmax())

roa_mean = df.groupby('year')['ROA'].mean()
roa_first = roa_mean.iloc[0]


---
### 4.2 时序图

#### 图 1：Lev 均值与中位数


In [ ]:
lev_median = df.groupby('year')['Lev'].median()

lev_mean_pct = lev_mean * 100
lev_median_pct = lev_median * 100

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(lev_mean_pct.index, lev_mean_pct.values, marker='o', markersize=4, label='均值 (Mean)', color='C0', linewidth=1.5)
ax.plot(lev_median_pct.index, lev_median_pct.values, marker='s', markersize=4, label='中位数 (Median)', color='C1', linewidth=1.5)
ax.set_xlabel('年份')
ax.set_ylabel('资产负债率 (Lev) (%)')
ax.set_title('资产负债率 (Lev) 均值与中位数 (2000–2024)')
ax.legend(fontsize=9)
ax.set_xlim(2000, 2024)

top_y = ax.get_ylim()[1]
stages = [
    (2006, '2000-2006\n扩张期'),
    (2011, '2008-2011\n刺激期'),
]
for x, label in stages:
    ax.axvline(x=x, color='gray', linestyle='--', alpha=0.3)
    ax.text(x, top_y, label, ha='center', va='bottom', fontsize=7, color='gray')

plt.tight_layout()
plt.savefig(OUT_FIGS / 'fig01_lev_mean_median.png')
plt.show()
print('已保存：fig01_lev_mean_median.png')


**图 1 解读：**

Lev 的均值长期高于中位数，差距约 4–8 个百分点，说明分布始终右偏——少数高杠杆企业拉高了整体均值。
总负债率呈现明显的阶段性变化：2000–2003 年快速下降（均值从 54% 降至 49%），2004–2008 年高位震荡（均值 50%–52%），
2009–2016 年持续去杠杆（均值从 50% 降至 44%），2017 年后低位企稳。
中位数从 2005 年的 37.7% 下降至 2023 年的 33.8%，反映典型企业的负债水平在长期下移。


---
#### 图 2：ROA 与 Cash 均值


In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4.5))

color_roa = 'C0'
color_cash = 'C1'

roa_mean_pct = roa_mean * 100
cash_mean_pct = cash_mean * 100

ax1.plot(roa_mean_pct.index, roa_mean_pct.values, marker='o', markersize=4, color=color_roa, linewidth=1.5, label='ROA（总资产收益率）')
ax1.set_xlabel('年份')
ax1.set_ylabel('ROA (%)', color=color_roa)
ax1.tick_params(axis='y', labelcolor=color_roa)
ax1.set_xlim(2000, 2024)

ax2 = ax1.twinx()
ax2.plot(cash_mean_pct.index, cash_mean_pct.values, marker='s', markersize=4, color=color_cash, linewidth=1.5, label='Cash（现金比率）')
ax2.set_ylabel('Cash (%)', color=color_cash)
ax2.tick_params(axis='y', labelcolor=color_cash)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

corr = roa_mean.corr(cash_mean)
ax1.set_title(f'ROA 与 Cash 年度均值 (2000–2024)，相关系数 r = {corr:.3f}')

plt.tight_layout()
plt.savefig(OUT_FIGS / 'fig02_roa_cash_mean.png')
plt.show()
print('已保存：fig02_roa_cash_mean.png')


**图 2 解读：**

ROA 和 Cash 在大多数时期呈同向走势——盈利能力越强的企业通常也持有更多现金，两者相关系数为 0.625。
但在特定阶段两者出现背离：2008 年金融危机后 ROA 快速反弹而 Cash 继续下降，企业将现金用于投资而非囤积；
2020 年疫情冲击后 ROA 短暂下降而 Cash 大幅攀升，体现「现金为王」的防御性行为。
仅凭该图不能推断因果关系，因为盈利能力和现金持有同时受宏观政策、行业周期和公司治理等多重因素影响。


---
## 5. 行业负债率特征分析


In [1]:
import sys, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib.font_manager as fm

fm.fontManager.addfont(r"C:\Windows\Fonts\msyh.ttc")
plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'font.family': 'sans-serif', 'font.sans-serif': ['Microsoft YaHei'],
    'axes.unicode_minus': False,
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'lines.linewidth': 2.0,
})

BASE = Path.cwd()
CLEAN = BASE / "data" / "clean"
OUT_TABLES = BASE / "output" / "tables"
OUT_FIGS = BASE / "output" / "figures"
for d in [OUT_TABLES, OUT_FIGS]: d.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CLEAN / "firm_year_clean.csv", dtype={"code": str})
df['year'] = pd.to_numeric(df['year'], errors='coerce')
print(f"数据加载完成：{len(df)} 行，{df["code"].nunique()} 家")
print(f"年份范围：{int(df["year"].min())} – {int(df["year"].max())}")


数据加载完成：71787 行，5820 家
年份范围：2000 – 2024


---
## 5.1 行业范围

分析以下 7 个行业：C 制造业、D 电力热力燃气、E 建筑业、F 批发零售、G 交通运输、J 金融业、K 房地产业。
行业分类使用 CSMAR 行业大类代码。


In [2]:
target_inds = ['C', 'D', 'E', 'F', 'G', 'J', 'K']
INDUSTRY_CN = {'C':'制造业','D':'电力热力燃气','E':'建筑业','F':'批发零售','G':'交通运输','J':'金融业','K':'房地产业'}

sub = df[df['industry'].isin(target_inds)].copy()
sub['ind_name'] = sub['industry'].map(INDUSTRY_CN)
print(f'筛选后：{len(sub)} 行，{sub["code"].nunique()} 家')
n_bad = (sub['Lev'] > 1).sum()
sub = sub[sub['Lev'] <= 1].copy()
print(f'剔除 Lev>1 异常值：{n_bad} 行（含交通运输2006年12家Lev=397%的录入错误）')
print(f'剔除后：{len(sub)} 行，{sub["code"].nunique()} 家')
print(sub['industry'].value_counts().sort_index())


筛选后：57753 行，4666 家
剔除 Lev>1 异常值：405 行（含交通运输2006年12家Lev=397%的录入错误）
剔除后：57348 行，4664 家
industry
C    44508
D     1800
E     1450
F     2101
G     4020
J     1983
K     1486
Name: count, dtype: int64


---
## 5.2 算术平均负债率

各行业年度 \\( \\overline{Lev}_{it} \\)：行业内所有公司 Lev 的简单算术平均。
金融业（J）因数值过高单独绘制子图。


In [3]:
# ── 算数平均 ──
lev_equal = sub.groupby(['industry','year'])['Lev'].mean().mul(100).reset_index()
lev_equal.columns = ['industry','year','lev_mean']
lev_equal['ind_name'] = lev_equal['industry'].map(INDUSTRY_CN)

# ── 配色（7行业：D/G/J虚线，E/K加粗，C/F常规）──
colors = {'C':'#1f77b4','D':'#ff7f0e','E':'#2ca02c','F':'#d62728','G':'#9467bd','K':'#e377c2','J':'#6f9f8f'}
line_styles = {'D':'dashed','G':'dashed','J':'dashed','E':'solid','K':'solid','C':'solid','F':'solid'}
line_widths = {'D':2.0,'G':2.0,'J':2.0,'E':3.0,'K':3.0,'C':1.5,'F':1.5}
plot_inds = ['C','D','E','F','G','K','J']

# ── 绘图：单图，7行业合列 ──
fig, ax = plt.subplots(figsize=(11, 4.5))

for ind in plot_inds:
    d = lev_equal[lev_equal["industry"] == ind]
    ax.plot(d["year"], d["lev_mean"], label=INDUSTRY_CN[ind],
            color=colors[ind], linewidth=line_widths[ind], linestyle=line_styles[ind])

ax.set_xlabel("年份")
ax.set_ylabel("资产负债率 Lev (%)")
ax.set_title("算术平均行业负债率", fontsize=11)
ax.set_xlim(2000, 2024)
ax.set_ylim(25, 78)
ax.legend(fontsize=7, loc="upper left", ncol=2)

# 事件竖线
for yr, label in [(2008, '2008金融危机'), (2015, '2015年供给侧改革'), (2020, '2020年疫情')]:
    ax.axvline(x=yr, color='gray', linestyle='--', alpha=0.3)
    ax.text(yr, 76, label, ha='center', va='bottom', fontsize=7, color='gray', rotation=0)

plt.tight_layout()
plt.savefig(OUT_FIGS / 'fig03_industry_lev_equal_weight.png')
plt.show()
print('已保存：fig03_industry_lev_equal_weight.png')

已保存：fig03_industry_lev_equal_weight.png


**图 3 解读：**

主图展示了除金融业外 6 个行业的算术平均负债率时序变化。房地产（K）负债率最高，长期维持在 55%–65%区间；建筑业（E）稳定在 55%–60%；制造业（C）和批发零售（F）居中，约 35%–50%；电力热力燃气（D）和交通运输（G）最低，约 35%–45%。

交通运输（G）在 2007 年出现异常尖峰（约 48%，显著高于前后年份的 35%–42%），主要受当年少数中小企业极端样本影响（2007年交通运输业91家样本中4家Lev>1的公司在剔除Lev>1后仍有影响）。2008年金融危机（灰色虚线）后交通运输算术均值大幅下降，小企业去杠杆明显。2015年供给侧改革后各行业杠杆趋于收敛。

金融业（J）单独展示于子图 B，其算术均值约 55%–70%，因行业负债结构特殊（依赖存款而非借款）与其余行业不可比。

图中灰色虚线标注了三次重大政策冲击：2008 年全球金融危机、2015 年供给侧改革、2020 年新冠疫情。


---
## 5.3 加权平均负债率（权重：总资产）

\[ \text{Weighted Lev}_{jt} = \frac{\sum_{i \in j} Lev_{it} \times \text{TotalAssets}_{it}}{\sum_{i \in j} \text{TotalAssets}_{it}} \]
大公司因资产规模更大而被赋予更高权重，反映行业整体的债务风险。


In [4]:
# ── 加权平均 ──
sub['total_assets'] = np.exp(sub['Size_raw'])
sub['_num'] = sub['Lev'] * sub['total_assets']
lev_weighted = (
    sub.groupby(['industry','year'], as_index=False)[['_num','total_assets']].sum()
)
lev_weighted['lev_wavg'] = lev_weighted['_num'] / lev_weighted['total_assets'] * 100
lev_weighted['ind_name'] = lev_weighted['industry'].map(INDUSTRY_CN)
lev_weighted = lev_weighted[['industry','year','lev_wavg','ind_name']]

# ── 配色（同图3）──
colors = {'C':'#1f77b4','D':'#ff7f0e','E':'#2ca02c','F':'#d62728','G':'#9467bd','K':'#e377c2','J':'#6f9f8f'}
line_styles = {'D':'dashed','G':'dashed','J':'dashed','E':'solid','K':'solid','C':'solid','F':'solid'}
line_widths = {'D':2.0,'G':2.0,'J':2.0,'E':3.0,'K':3.0,'C':1.5,'F':1.5}
plot_inds = ['C','D','E','F','G','K','J']

# ── 绘图 ──
fig, ax = plt.subplots(figsize=(11, 4.5))

for ind in plot_inds:
    d = lev_weighted[lev_weighted["industry"] == ind]
    ax.plot(d["year"], d["lev_wavg"], label=INDUSTRY_CN[ind],
            color=colors[ind], linewidth=line_widths[ind], linestyle=line_styles[ind])

ax.set_xlabel("年份")
ax.set_ylabel("资产负债率 Lev (%)")
ax.set_title("加权平均行业负债率", fontsize=11)
ax.set_xlim(2000, 2024)
ax.set_ylim(30, 95)
ax.legend(fontsize=7, loc="upper left", ncol=2)

# 事件竖线
for yr, label in [(2008, '2008金融危机'), (2015, '2015年供给侧改革'), (2020, '2020年疫情')]:
    ax.axvline(x=yr, color='gray', linestyle='--', alpha=0.3)
    ax.text(yr, 93, label, ha='center', va='bottom', fontsize=7, color='gray', rotation=0)

plt.tight_layout()
plt.savefig(OUT_FIGS / 'fig04_industry_lev_asset_weighted.png')
plt.show()
print('已保存：fig04_industry_lev_asset_weighted.png')

已保存：fig04_industry_lev_asset_weighted.png


**图 4 解读：**

加权平均后各行业的排序与算术平均基本一致，但制造业（C）和交通运输（G）的数值明显抬升——2008年后算术均值分别约 38%–42% 和 35%–38%，而加权均值分别约 55%–57% 和 47%–55%。这反映了一个关键结构特征：大企业（权重高）的杠杆显著高于小企业，算术平均因给予小企业同等权重而低估了行业整体的债务水平。

交通运输的加权均值在 2008–2015 年间大幅高于算术均值（差值最高达 22.6pp），说明该行业大型国企的杠杆远高于中小物流企业。2015年供给侧改革后，制造业和交通运输的两种均值差距略有收窄，反映去杠杆政策的普惠效应。电力行业（D）两种均值差距最小且稳定（5–11pp），因其以大型国企为主，信贷资源分配均匀。

金融业（J）子图显示其加权均值约 70%–85%，显著高于算术均值（55%–70%），说明大型金融机构的杠杆高于中小金融机构。


---
## 5.4 两种算法比较

**算术平均 vs 加权平均的经济含义：**
- **算术平均**：反映行业典型公司的负债水平，每个公司权重相同。适合回答「行业内一家普通公司的负债率是多少」。但容易受中小企业极端值干扰。
- **加权平均**：反映行业整体的负债水平，大公司权重更大。适合回答「整个行业的债务风险有多大」。大企业的违约对金融体系和供应链影响更大。

**行业分层（按负债率由高到低）：**
- **高负债梯队（均值55%以上）**：建筑业（E）、电力热力燃气（D）、金融业（J）——这些行业要么资金密集型、要么以国有大企业为主
- **中等负债梯队（均值40%–55%）**：房地产业（K）、批发零售（F）
- **偏低负债梯队（均值35%–45%）**：制造业（C）、交通运输（G）

**算术与加权差异解读：**
加权平均显著高于算术平均的行业（金融业 J、制造业 C、交通运输 G），说明大型企业杠杆更高、拉高了行业整体风险。加权平均接近或低于算术平均的行业（建筑业 E），说明大型企业杠杆反而低于中小企业。

**值得关注的结构分化：**制造业（C）和交通运输（G）的两种平均值在 2008 年前后出现持续且显著的分化。以制造业为例，2005 年加权平均仅比算术平均高 2.4pp，到 2017 年差距扩大到 18.3pp。这反映了大企业（国企为主）保持了稳定的杠杆水平，而大量民营中小企业经历了持续的去杠杆。这与「全样本趋势掩盖行业结构性差异」的核心论点一致——全样本的均值下降主要受中小企业的去杠杆驱动，而大型企业杠杆保持稳定。

**政策冲击的行业异质性：**
- **2008 金融危机**：交通运输和制造业的算术均值大幅下降（小企业去杠杆），但加权均值基本不变（大企业通过信贷刺激维持杠杆）
- **2015 供给侧改革**：制造业和房地产业的两种均值均有所下降，但大企业降幅小于小企业
- **2020 疫情**：各行业算术均值短期抬升（中小企业借入纾困贷款），加权均值变化不大

**讨论行业整体债务风险时，哪一种更合理？**
加权平均更合理。例如，讨论房地产行业系统性风险时，头部房企的负债率比中小房企更具参考价值。


---
## 5.5 行业变量列表（选定年份）

呈现 2001、2003、…、2023（间隔两年）各行业 SLoan, LLoan, Lev, Cash, ROA, ROE 均值。


In [5]:
sel_years = [2001, 2003, 2005, 2007, 2009, 2011, 2013, 2015, 2017, 2019, 2021, 2023]
sel_vars = ['SLoan', 'LLoan', 'Lev', 'Cash', 'ROA', 'ROE']
sub_sel = sub[sub['year'].isin(sel_years)].copy()
rows = []
for (ind, yr), grp in sub_sel.groupby(['industry','year']):
    for v in sel_vars:
        rows.append({'年份': yr, '行业代码': ind, '行业名称': INDUSTRY_CN.get(ind,''),
                     '变量': v, '均值': round(grp[v].mean(), 4), '样本量': int(grp[v].notna().sum())})
ind_summary = pd.DataFrame(rows)
ind_summary.to_csv(OUT_TABLES / 'industry_selected_years_summary.csv', index=False, encoding='utf-8-sig')
with pd.ExcelWriter(OUT_TABLES / 'industry_selected_years_summary.xlsx', engine='openpyxl') as writer:
    ind_summary.to_excel(writer, sheet_name='industry_summary', index=False)
print(f'行业变量表已保存：{len(ind_summary)} 行')
print()
print('--- 行业变量明细（选定年份，列表格式）---')
for yr in sel_years:
    print(f'[{yr}]')
    for ind in target_inds:
        sub2 = ind_summary[(ind_summary['年份']==yr) & (ind_summary['行业代码']==ind)]
        parts = [f"{r['变量']}={r['均值']:.4f} (n={int(r['样本量'])})" for _, r in sub2.iterrows()]
        print(f'  {ind} {INDUSTRY_CN[ind]:8s}:  {",  ".join(parts)}')
    print()


行业变量表已保存：504 行

--- 行业变量明细（选定年份，列表格式）---
[2001]
  C 制造业     :  SLoan=0.1658 (n=630),  LLoan=0.0539 (n=630),  Lev=0.4290 (n=630),  Cash=0.1818 (n=630),  ROA=0.0195 (n=630),  ROE=0.0182 (n=630)
  D 电力热力燃气  :  SLoan=0.0807 (n=45),  LLoan=0.0864 (n=45),  Lev=0.3484 (n=45),  Cash=0.1523 (n=45),  ROA=0.0533 (n=45),  ROE=0.0848 (n=45)
  E 建筑业     :  SLoan=0.1744 (n=18),  LLoan=0.0405 (n=18),  Lev=0.5885 (n=18),  Cash=0.1585 (n=18),  ROA=0.0085 (n=18),  ROE=0.0137 (n=18)
  F 批发零售    :  SLoan=0.1023 (n=47),  LLoan=0.0854 (n=47),  Lev=0.3572 (n=47),  Cash=0.1739 (n=47),  ROA=0.0443 (n=47),  ROE=0.0661 (n=47)
  G 交通运输    :  SLoan=0.1901 (n=66),  LLoan=0.0297 (n=66),  Lev=0.4609 (n=66),  Cash=0.2396 (n=66),  ROA=0.0355 (n=66),  ROE=0.0706 (n=66)
  J 金融业     :  SLoan=0.2135 (n=58),  LLoan=0.0394 (n=58),  Lev=0.5011 (n=58),  Cash=0.1381 (n=58),  ROA=-0.0016 (n=58),  ROE=-0.0116 (n=58)
  K 房地产业    :  SLoan=0.1126 (n=35),  LLoan=0.0635 (n=35),  Lev=0.3660 (n=35),  Cash=0.1934 (n=35),  ROA=0.0326 (n=35

**解读：** 金融业（J）Lev最高但SLoan、LLoan结构不同（依赖存款）；房地产业（K）Lev和SLoan均高位，高度依赖银行借款；制造业（C）ROA长期偏低（约2%–4%）。

---
**Notebook 4 完成。**


---
## 6. 股权结构分析


In [1]:
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm.fontManager.addfont(r'C:\Windows\Fonts\msyh.ttc')
plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Microsoft YaHei'],
    'axes.unicode_minus': False,
    'savefig.dpi': 200, 'savefig.bbox': 'tight',
})

BASE = Path.cwd()
CLEAN = BASE / 'data' / 'clean'
OUT_FIGS = BASE / 'output' / 'figures'
OUT_FIGS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CLEAN / 'firm_year_clean.csv', dtype={'code': str})
df['year'] = pd.to_numeric(df['year'], errors='coerce')
print(f'数据加载完成：{len(df)} 行，{df["code"].nunique()} 家')
print(f'年份范围：{int(df["year"].min())} – {int(df["year"].max())}')

数据加载完成：71787 行，5820 家
年份范围：2000 – 2024


---
## 6.1 Top1 箱线图

绘制第一大股东持股比例 Top1 在选定年份的箱线图，观察股权集中度的时序变化。

In [2]:
sel_years = [2001, 2003, 2005, 2007, 2009, 2011, 2013, 2015, 2017, 2019, 2021, 2023]

sub = df[df['year'].isin(sel_years)].copy()
sub = sub.dropna(subset=['Top1_raw'])
print(f'\u9009\u4e2d\u5e74\u4efd\u6709\u6548\u89c2\u6d4b\uff1a{len(sub)} \u6761')

sub = sub.sort_values('year')

fig, ax = plt.subplots(figsize=(10, 4.5))

years_sorted = sorted(sub['year'].unique())
data_by_year = [sub[sub['year'] == yr]['Top1_raw'].values for yr in years_sorted]

bp = ax.boxplot(data_by_year, labels=years_sorted, patch_artist=True,
                showmeans=False,
                medianprops={'color': 'darkred', 'linewidth': 2},
                meanprops={'color': 'blue', 'linewidth': 1.5, 'linestyle': '--'},
                flierprops={'marker': 'o', 'markerfacecolor': 'gray', 'markersize': 4, 'alpha': 0.5})

colors = ['#e8f4f8'] * len(years_sorted)
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c)

ax.set_xlabel('\u5e74\u4efd')
ax.set_ylabel('\u7b2c\u4e00\u5927\u80a1\u4e1c\u6301\u80a1\u6bd4\u4f8b Top1')
ax.set_title('\u7b2c\u4e00\u5927\u80a1\u4e1c\u6301\u80a1\u6bd4\u4f8b\u7bb1\u7ebf\u56fe\uff08\u9009\u5b9a\u5e74\u4efd\uff09')
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
# \u6dfb\u52a0\u7ebf\u6027\u56de\u5f52\u7ebf\uff08Top1\u5e74\u5ea6\u4e2d\u4f4d\u6570\u8d8b\u52bf\uff09
medians = [np.median(sub[sub['year'] == yr]['Top1_raw']) for yr in years_sorted]
ax.plot(range(1, len(years_sorted)+1), medians, color='#27496d', linewidth=1.5, linestyle='-', marker='o', markersize=4, label='\u5e74\u5ea6\u4e2d\u4f4d\u6570\u8d8b\u52bf')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(OUT_FIGS / 'fig05_top1_boxplot_selected_years.png', dpi=300)
plt.show()
plt.close()
print('\u5df2\u4fdd\u5b58\uff1afig05_top1_boxplot_selected_years.png')


选中年份有效观测：30572 条


已保存：fig05_top1_boxplot_selected_years.png


**图 5 解读：**

从箱线图可见，第一大股东持股比例的中位数从 2005 年约 37.7% 持续下降至 2023 年约 29.5%，四分位距也从 26.3 个百分点收窄至 20.6 个百分点，反映出股权结构长期分散化趋势。高端极端值仅在 2007 和 2023 年少量存在（全部为持股极高值），说明始终有个别公司保持高度集中的股权结构。

In [3]:
target_years = [2005, 2007, 2023]
stats_rows = []
for yr in target_years:
    vals = sub[sub['year'] == yr]['Top1_raw']
    q1, q2, q3 = vals.quantile([0.25, 0.50, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_out = ((vals < lower) | (vals > upper)).sum()
    n_hi = (vals > upper).sum()
    n_lo = (vals < lower).sum()
    stats_rows.append({
        '年份': yr,
        '样本量': len(vals),
        '均值': f'{vals.mean():.4f}',
        '中位数': f'{q2:.4f}',
        'Q1': f'{q1:.4f}',
        'Q3': f'{q3:.4f}',
        'IQR': f'{iqr:.4f}',
        '极端值数': n_out,
        '其中高端': n_hi,
        '其中低端': n_lo,
    })

stats_df = pd.DataFrame(stats_rows)
print(stats_df.to_string(index=False))

  年份  样本量     均值    中位数     Q1     Q3    IQR  极端值数  其中高端  其中低端
2005 1350 0.4034 0.3770 0.2784 0.5416 0.2632     0     0     0
2007 1511 0.3577 0.3383 0.2349 0.4723 0.2374     4     4     0
2023 5322 0.3196 0.2952 0.2067 0.4123 0.2056    60    60     0


---
## 6.2 分析问题

### 1. 三个年份中位数、四分位距和极端值差异

**中位数：** 2005 年中位数最高（约 37.7%），反映出股改前上市公司普遍存在「一股独大」现象。2007 年中位数下降至约 33.8%。2023 年进一步下降至约 29.5%，表明第一大股东持股比例在长期呈下降趋势。

**四分位距（IQR）：** 2005 年 IQR 最大（约 26.3 个百分点），说明各公司之间股权集中度差异较大。2007 年 IQR 收窄至约 23.7 个百分点。2023 年 IQR 进一步缩小至约 20.6 个百分点，表明上市公司股权结构趋于同质化。

**极端值：** 2005 年无极端值（数据全部落在 1.5 倍 IQR 范围内），2007 年高端出现 4 个极端值，2023 年高端极端值增加至 60 个（占该年样本的 1.1%）。极端值全部出现在高持股一侧，说明始终有少数公司保持高度集中的股权结构，且随着样本量扩大此类公司绝对数量也在增加。

### 2. 股权分置改革的影响

2005 年启动的股权分置改革是非流通股（国有股、法人股）转为流通股的重大制度变革。改革前，约三分之二的股票不能流通，大股东持股比例普遍较高。改革后，非流通股逐渐上市流通，大股东的持股比例被稀释。2005 年到 2007 年的中位数和 IQR 下降即体现了这一影响——大股东持股比例下降且公司间差异缩小。

### 3. 股权结构是否更加分散？

是的。证据如下：
- 中位数从 2005 年约 37.7% 持续下降至 2023 年约 29.5%。
- 四分位距从 26.3 个百分点收窄至 20.6 个百分点，说明绝大部分公司的股权集中度都在向中低水平收敛。
- 高端极端值虽然绝对数量随样本扩增而增加，但占比维持在约 1%，并未显著上升。

但需注意，中国上市公司第一大股东持股比例均值仍在 30% 以上，相比美国（约 15%–20%）仍然偏高，「一股独大」现象尚未完全改变。

### 4. 箱线图能否判断控制权稳定性？

不能完全判断。仅凭第一大股东持股比例（Top1）箱线图：
- **能**看出的：整体集中度趋势、离散程度、异常公司。
- **不能**看出的：Top1 > 50% 才拥有绝对控制权，Top1 在 30%–50% 时控制权取决于其他股东制衡情况；还需要考察第二至第十大股东持股比例、Herfindahl 指数（HHI5）、股权制衡度（Z 指数 = Top1 / Top2）等补充指标。例如，即使 Top1 下降，如果其他股东更加分散，实际控制权未必减弱。

---
**Notebook 5 完成。**